<a href="https://colab.research.google.com/github/NairaGama/An-lise-de-Tempo-de-Entrega-em-Delivery/blob/main/An%C3%A1lisedeTempodeEntrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Análise de Tempo de Entrega em Operações de Delivery**

**Autora**: Naira Gama

**Informações gerais**

|Base de dados| [Link](https://www.kaggle.com/datasets/denkuznetz/food-delivery-time-prediction/data)|
|:------|:-----|
|Nome da base| “Food Delivery Time Prediction”|
|Origem| Kaggle.com|


##**Descrição do Problema**

O tempo de entrega é um dos principais desafios nas operações de delivery. Esse processo envolve as etapas de preparo do pedido, embalagem, despacho e transposte até o cliente. Há diversos fatores que podem influenciar no tempo de entrega, como a distância, as condições de tráfego, o horário que o pedido foi realizado, e o tipo de veículo utilizado na entrega.
Nesse contexto, a análise de dados torna-se essencial para identificar as variáveis que estão associadas ao aumento ou redução do tempo, permitindo assim a geração de insights que podem contribuir para melhorias.

##**Objetivo**

O principal objetivo deste projeto é a realização de uma análise exploratória dos dados, visualizando assim os fatores importantes para redução do tempo de entrega em operações de delivery.

##**Tarefas Realizadas**

*   Carregamento automático dos dados
*   Renomeação de colunas
*   Pré-análise
*   Limpeza dos dados
*   Transformação dos dados
*   Análise Exploratória dos Dados (EDA)
*   Insights da análise
*   Análise de interação entre fatores
*   Dataset final para dashboard


#**Bibliotecas**


In [ ]:
#Bibliotecas utilizadas
import pandas as pd #manipulação e análise
import numpy as np #análise

#Geração de gráficos
import matplotlib.pyplot as plt
import seaborn as sns

#Para acessar dataset
import kagglehub

from kagglehub import KaggleDatasetAdapter

#**Carregamento automático dos Dados**

Nesta etapa é realizado o carremento automático dos dados, diretamente do Kaggle, assim não é necessária a realização de download manual do arquivo na máquina.
As cinco primeiras linhas do dataset são exibidas no final desta etapa, o que permite a visualização da estrutura.

In [ ]:
#Carregamento automático

df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS,
                            "denkuznetz/food-delivery-time-prediction",
                            "Food_Delivery_Times.csv")

#Exibição das 5 primeiras linhas
print(df.head())

#**Renomeação de colunas**

Nesta etapa, acontece a renomeação de colunas, assim facilita-se a leitura e a compreensão dos dados para aqueles que não dominam a língua inglesa.

In [ ]:
#Renomeação das colunas
df.columns = ['id', 'distancia_km', 'clima', 'trafego', 'horario', 'tipo_veiculo', 'tempo_preparacao', 'experiencia_entregador', 'tempo_entrega' ]
df

Também é necessária a tradução dos valores categóricos, assim mantém-se a consistência com os nomes das variávies e facilita a interpretação da análise.

In [ ]:
#verificando valores únicos antes da tradução

valores_unicos = pd.DataFrame({
    'clima': pd.Series(df['clima'].unique()),
    'trafego': pd.Series(df['trafego'].unique()),
    'horario': pd.Series(df['horario'].unique()),
    'tipo_veiculo': pd.Series(df['tipo_veiculo'].unique())
})
valores_unicos

In [ ]:
#Tradução: inglês para português

#clima
map_clima = {
    'Windy': 'Ventoso',
    'Clear': 'Claro',
    'Foggy': 'Enevoado',
    'Rainy': 'Chuvoso',
    'Snowy': 'Nevado'
}
df['clima']= df['clima'].replace(map_clima)


In [ ]:
#trafego
map_trafego = {
    'Low': 'Baixo',
    'Medium': 'Médio',
    'High': 'Alto'
}
df['trafego']= df['trafego'].replace(map_trafego)

In [ ]:
#horario
map_horario = {
    'Afternoon': 'Tarde',
    'Evening': 'Noite',
    'Night': 'Noite',
    'Morning': 'Manhã'
}
df['horario']= df['horario'].replace(map_horario)

In [ ]:
map_tipo_veiculo= {
    'Scooter': 'Motocicleta',
    'Bike': 'Bicicleta',
    'Car': 'Carro'
}
df['tipo_veiculo']= df['tipo_veiculo'].replace(map_tipo_veiculo)

In [ ]:
#verificação após a tardução
df.head()

#**Dicionário**

A tabela a seguir apresenta o dicionário de dados, contemplando as variáveis originais, as variáveis adotadas, tipo de dado e a sua  descrição. Este dicionário tem como objetivo garantir a padronização e compreensão na documentação dos dados.

|Campo original|Campo adotado|Tipo|Descrição|
|:-------------|:------------|:--------|:---------|
|Order_ID|id|int|Identificador único de cada pedido|
|Distance_km|distancia_km|float|Distância da entrega em quilômetros|
|Weather|clima|string|Condições meteorológicas durante a entrega do pedido|
|Traffic_Level|trafego|string|Condições de tráfego durante a entrega|
|Time_of_Day|horario|string|Horário da entrega (manhã, tarde, noite ou madrugada)|
|Vehicle_Type|tipo_veiculo|string|Tipo de veículo usado para realizar a entrega|
|Preparation_Time_min|tempo_preparacao|int|Tempo de prepação do pedido em minutos|
|Courier_Experience_yrs|experiencia_entregador|float|Tempo de experiência do entregador em anos|
|Delivery_Time_min|tempo_entrega|int|Tempo total de entrega em minutos|

#**Pré-análise(verificação dos dados)**

Esta é a etapa de realização de análise para entendimento da estrutura do dataset.

In [ ]:
#Verificando o volume
df.shape

In [ ]:
#Verificação de dados ausentes
df.isnull().sum()

Com esta análise, identificou-se:


*   1.000 registros
*   9 variáveis
*   Valores ausentes em 4 colunas (clima, trafego, horario e experiencia_entregador).




In [ ]:
#Proporção de valores ausentes
(df.isnull().sum()/len(df))*100

A porcentagem de valores ausentes em cada uma das 4 colunas é de 3%.
A solução para este problema é:  

*   nas variáveis categóricas, substituir pela moda
*   na variável numérica, usar a mediana

#**Limpeza dos Dados**

Nesta etapa acontece a limpeza dos dados, e no caso deste dataset, os valores ausentes foram substituídos pela moda e pela mediana.

In [ ]:
#Trantando as variáveis categóricas (preenchendo com a moda)
df['clima'].fillna(df['clima'].mode()[0], inplace=True)
df['trafego'].fillna(df['trafego'].mode()[0], inplace=True)
df['horario'].fillna(df['horario'].mode()[0], inplace=True)

In [ ]:
#Tratando a variável numérica (Preenchendo com a mediana)
df['experiencia_entregador'].fillna(df['experiencia_entregador'].median(), inplace=True)

In [ ]:
#Verificando valores ausentes após a limpeza
df.isnull().sum()

Após a limpeza, não encontra-se mais nenhum valor ausente.

####**Registros duplicados**

Aqui é onde verifica-se a existência de valores duplicados. Se houver, a limpeza é realizada e as linhas duplicadas podem ser removidas.
Com esta etapa, evita-se erros estatísticos e garantimos a qualidade dos dados.
Não foi encontrado nenhum registro duplicado neste dataset.

In [ ]:
registros_duplicados = df.duplicated().sum()
print(f"Número de registros duplicados no dataset: {registros_duplicados}")

#**Transformação dos Dados**

Aqui são criadas as variáveis derivadas, categoria distância, que indica se ela é curta, média ou longa. Com a transformação, o objetivo é facilitar a análise e a construção do dashboard final.

In [ ]:
#Categoria distância
bins = [0, 3, 7, np.inf]
labels = ['curta', 'média', 'longa']

df['categoria_distancia'] = pd.cut(df['distancia_km'], bins=bins, labels=labels, right=False)

A variável distância_km foi categorizada em três faixas: curta, média e longa. Isso facilita a interpretação da análise.

In [ ]:
#Categoria preparo
bins = [0, 15, 30, np.inf]
labels = ['rápido', 'médio', 'lento']

df['categoria_preparo'] = pd.cut(df['tempo_preparacao'], bins=bins, labels=labels, right=False)

A variável 'categoria_preparo' foi categorizada nas faixas: rápido, médio e lento.

In [ ]:
#Nível de experiência
bins = [0, 2, 5, np.inf]
labels = ['iniciante', 'intermediário', 'experiente']

df['nivel_experiencia'] = pd.cut(df['experiencia_entregador'], bins=bins, labels=labels, right=False)

O nível de experiência foi categorizado em: iniciante, intermediário e experiente.

In [ ]:
#Categoria entrega
bins = [0, 30, 60, np.inf]
labels = ['curto', 'médio', 'longo']

df['categoria_entrega'] = pd.cut(df['tempo_entrega'], bins=bins, labels=labels, right=False)

O tempo de entrega foi categorizado em: curto, médio e longo.

In [ ]:
#Validando a categorização
df[['distancia_km', 'categoria_distancia','tempo_preparacao', 'categoria_preparo', 'experiencia_entregador', 'nivel_experiencia', 'tempo_entrega', 'categoria_entrega']].head()

#**Análise Exploratória dos Dados (EDA)**

A etapa de análise exploratória tem o objetivo de entender os dados, verificar possíveis padrões e problemas.

####**Estrutura Geral do DataFrame**

In [ ]:
df

####**Estatísticas**

Nesta etapa, verifica-se as estatísticas do dataset, como valores mínimo e máximo das variáveis numéricas, e a frequência no caso das variáveis categóricas.
Como o objetivo é reduzir o tempo de entrega, também verificou-se a média desse tempo em todas as variáveis que podem influenciá-lo.

In [ ]:
#Estatística Descritiva
df.describe()

In [ ]:
#Estatística das variáveis categóricas
df.describe(include=['object'])

In [ ]:
#Média do tempo de entrega de acordo a distância
df.groupby('categoria_distancia', observed=True)['tempo_entrega'].mean()

Entregas com **longa** distância apresentam maior tempo médio de entrega.

In [ ]:
#Média do tempo de entrega de acordo o clima
df.groupby('clima')['tempo_entrega'].mean()

Entregas realizadas durante o clima **Nevado** apresentam maior tempo médio de entrega.

In [ ]:
#Média do tempo de entrega de acordo o tráfego
df.groupby('trafego')['tempo_entrega'].mean()

Entregas realizadas trafego estava **Baixo** apresentam menor tempo médio de entrega quando comparadas ao tráfego **Alto**.

In [ ]:
#Média do tempo de entrega de acordo o horário
df.groupby('horario')['tempo_entrega'].mean()

O tempo médio de entrega nos três horários do dia é bem similar, variando entre 56.08 durante a **tarde**, e 57.64 durante a **noite**.

In [ ]:
#Média do tempo dde entrega de acordo o tipo de veículo utilizado
df.groupby('tipo_veiculo')['tempo_entrega'].mean()

O tempo médio de entrega com os três tipos de veículo é similar, variando entre 56.04 quando realizado de **Motocicleta**, e 58.20 quando realiado com um **Carro**.

In [ ]:
#Média do tempo de entrega de acordo o tempo de preparo
df.groupby('categoria_preparo', observed=True)['tempo_entrega'].mean()

O tempo médio de entrega em preparos **rápidos** foi menor que nos preparos **médios**.

In [ ]:
#Média do tempo de entrega de acordo a experiência do entregador
df.groupby('nivel_experiencia', observed=True)['tempo_entrega'].mean()

O tempo médio de entrega realizada por profissionais **iniciantes** foi maior quando comparado aos **experientes**.

####**Tipos de Dados**

Aqui encontra-se os tipos de dados do dataframe.

In [ ]:
#Tipode de dados encontrados
print("Tipos de dados encontrados:")
df.dtypes

####**Valores Únicos**

Esta é a etapa de contagem de valores únicos, assim é possível verficar quantos valores diferentes existem em cada coluna do dataset. Por exemplo, nota-se que na coluna **trafego** há o valor 3, então isso indica que existem três possibilidades nesta coluna: Baixo, Médio ou Alto.

In [ ]:
#Valores únicos encontrados
print('Valores únicos:')
df.nunique()

####**Coluna chave**

A coluna chave é uma variável que identifica cada registro de forma única. No caso deste conjunto, a coluna chave é a id, pois cada pedido tem seu identificar único. Na célula de código a seguir, foi realizada essa verificação, e com isso confirma-se que não existem registros repetidos.

In [ ]:
df['id'].duplicated().sum()

####**Frequências**

Nesta etapa, observa-se a frequência de cada valor, ou seja, quantas vezes uma categoria aparece na coluna. Por exemplo, na coluna **categoria_preparo**, quantas vezes aparece o tempo médio?
A análise a seguir apresenta as frequências das seguintes colunas:

*   clima
*   trafego
*   horario
*   tipo_veiculo
*   categoria_distancia
*   categoria_preparo
*   nivel_experiencia
*   categoria_entrega


In [ ]:
#Distribuição das variáveis
#clima
df['clima'].value_counts()

In [ ]:
#Distribuição das variáveis
#trafego
df['trafego'].value_counts()

In [ ]:
#Distribuição das variáveis
#horario
df['horario'].value_counts()

In [ ]:
#Distribuição das variáveis
#tipo_veiculo
df['tipo_veiculo'].value_counts()

In [ ]:
#Distribuição das variáveis
#categoria_distancia
df['categoria_distancia'].value_counts()

In [ ]:
#Distribuição das variáveis
#categoria_preparo
df['categoria_preparo'].value_counts()

In [ ]:
#Investigando a ausência de casos de preparo lento
df['categoria_preparo'].value_counts(dropna=False)

In [ ]:
df[df['categoria_preparo'] == 'lento']['tempo_entrega']

###**<font color="yellow">Observação:</font>**

Após verificação, notou-se que não houve nenhum caso de preparo lento.


In [ ]:
#Distribuição das variáveis
#nivel_experiencia
df['nivel_experiencia'].value_counts()

In [ ]:
#Distribuição das variáveis
#categoria_entrega
df['categoria_entrega'].value_counts()

####**Análise da contagem:**

Com a verificação das frequências, nota-se que:

*   Em 50% das entregas (500) o clima foi classificado como claro
*   O tráfego variou, ma a maioria foi classificado como médio ou baixo
*   A maioria das entregas foram realizadas de bicicleta
*   A maioria das distâncias observadas foram longas
*   Não houve nenhum preparo lento (maior que 30 minutos)
*   Boa parcela dos entregadores são experientes
*   A maioria das entregas tiveram um tempo médio ou longo


####**Gráficos e distribuição entre variáveis**

Como o objetivo é a redução no tempo de entrega, a variável alvo é a **categoria_entrega**, e assim é possível realizar a análise de fatores que a influenciam.

####**Entrega X Distância**

In [ ]:
#Tabela de distribuição (categoria_entrega e categoria_distancia)
tab_dist = pd.crosstab(df['categoria_distancia'], df['categoria_entrega'])
tab_dist

In [ ]:
tab_dist_prop = pd.crosstab(df['categoria_distancia'], df['categoria_entrega'], normalize = 'index') * 100
tab_dist_prop.round(2)

In [ ]:
#Distribuição
#Gráfico categoria_entrega X categoria_distancia

sns.countplot(data=df, x='categoria_distancia', hue='categoria_entrega', palette=['#0AF567', '#FDA615', '#F50201'])

plt.title('Distância de Entrega por Distância')
plt.xlabel('Distância')
plt.ylabel('Quantidade')
plt.legend(title='Tempo de Entrega')
plt.show()

In [ ]:
#Proporção
plt.figure(figsize=(16, 8))

colors = ['#0AF567', '#FDA615', '#F50201']
tab_dist_prop.plot(kind='bar', stacked=True, color=colors)

plt.title('Proporção do Tempo de Entrega por Distância')
plt.xlabel('Categoria de Distância')
plt.ylabel('Percentual (%)')
plt.legend(title='Tempo de Entrega',
           bbox_to_anchor=(1.05, 1),
           loc='upper left')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Média
media_dist = df.groupby('categoria_distancia')['tempo_entrega'].mean()
media_dist

media_dist.plot(kind='bar', color=colors)

plt.title('Tempo Médio de Entrega por Distância')
plt.xlabel('Distância')
plt.ylabel('Tempo Médio (min)')
plt.xticks(rotation=0)
plt.show()

Observa-se que entregas com distâncias **curtas** concentram-se principalmente em tempos **curtos** e **médios**, já as distâncias **longas** têm maior quantidade de entregas de tempo **longo**.

O tempo médio de entrega cresce de acordo com a distância. Aproximadamente 31 minutos quando a distância é **curta**, 42 minutos para distância **média**, e mais de 60 minutos em distância **longa**. Ou seja, quanto maior é a distância, mais longa é a entrega.




####**Entrega X Clima**

In [ ]:
#Tabela de distribuição (categoria_entrega e clima)
tab_clima = pd.crosstab(df['clima'], df['categoria_entrega'])
tab_clima


In [ ]:
tab_clima_prop = pd.crosstab(df['clima'], df['categoria_entrega'], normalize = 'index') * 100
tab_clima_prop.round(2)

In [ ]:
#Gráfico categoria_entrega X clima
sns.countplot(data=df, x='clima', hue='categoria_entrega', palette=['#0AF567', '#FDA615', '#F50201'])

plt.title('Clima x Entrega')
plt.xlabel('Clima')
plt.ylabel('Quantidade')
plt.legend(title='Tempo de Entrega')
plt.show()

In [ ]:
#Proporção
plt.figure(figsize=(16, 8))

colors = ['#0AF567', '#FDA615', '#F50201']
tab_clima_prop.plot(kind='bar', stacked=True, color=colors)

plt.title('Proporção do Tempo de Entrega pelo Clima')
plt.xlabel('Clima')
plt.ylabel('Percentual (%)')
plt.legend(title='Tempo de Entrega',
           bbox_to_anchor=(1.05, 1),
           loc='upper left')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Média de tempo de entrega por clima
media_clima = df.groupby('clima')['tempo_entrega'].mean()
media_clima.round(2)


In [ ]:
#Boxplot
sns.boxplot(x='clima', y='tempo_entrega', data=df, color="#f4fa9c")

plt.title('Distribuição Tempo de Entrega por Clima')
plt.xlabel('Clima')
plt.ylabel('Tempo de Entrega (min)')
plt.xticks(rotation=0)
plt.show()

Observa-se que, nos cinco tipos clima, a maioria das entregas realizadas estão concentradas em duas categorias de tempo, o **médio** e **longo**.

Nota-se também, que há um desbalanceamento na quantidade de registros de entregas em dias com clima **Claro**, o que pode influenciar a análise baseada apenas em volume.

Verificando o tempo médio, nota-se que não há diferenças relevantes entre os climas, variando entre aproximadamente 53 minutos (no clima **Claro**) e 67 minutos (no clima **Nevado**). Com isso, percebe-se que o clima não é um fator determinante no aumento do tempo de entrega.

####**Entrega X Tráfego**

In [ ]:
#Tabela de distribuição (categoria_entrega e trafego)
tab_trafego= pd.crosstab(df['trafego'], df['categoria_entrega'])
tab_trafego

In [ ]:
tab_trafego_prop = pd.crosstab(df['trafego'], df['categoria_entrega'], normalize = 'index') * 100
tab_trafego_prop.round(2)

In [ ]:
#Distribuição
#Gráfico categoria_entrega X trafego
sns.countplot(data=df, x='trafego', hue='categoria_entrega', palette=['#0AF567', '#FDA615', '#F50201'])

plt.title('Trafego x Entrega')
plt.xlabel('Trafego')
plt.ylabel('Quantidade')
plt.legend(title='Tempo de Entrega')
plt.show()

In [ ]:
#Proporção
plt.figure(figsize=(16, 8))

colors = ['#0AF567', '#FDA615', '#F50201']
tab_trafego_prop.plot(kind='bar', stacked=True, color=colors)

plt.title('Proporção do Tempo de Entrega pelo Trafego')
plt.xlabel('Trafego')
plt.ylabel('Percentual (%)')
plt.legend(title='Tempo de Entrega',
           bbox_to_anchor=(1.05, 1),
           loc='upper left')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Média de tempo de entrega por Trafego
media_trafego = df.groupby('trafego')['tempo_entrega'].mean()
media_trafego.round(2)

In [ ]:
#Boxplot
sns.boxplot(x='trafego', y='tempo_entrega', data=df, color="#f4fa9c")

plt.title('Distribuição Tempo de Entrega por Trafego')
plt.xlabel('Trafego')
plt.ylabel('Tempo de Entrega (min)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Heatmap
tab_trafego = pd.crosstab(df['trafego'], df['categoria_entrega'], normalize ='index')
sns.heatmap(tab_trafego, annot=True, cmap='coolwarm')

Nota-se que em condições de **tráfego alto**, há predominância de entregas com tempo **longo**. Já em condições de **tráfego médio** e **baixo**, observa-se maior concentração de entregas com tempo **médio**, embora ainda exista uma parcela relevante de entregas **longas**.

O tempo médio de entrega varia aproximadamente entre 53 minutos em situações de tráfego **baixo** e 65 minutos em tráfego **alto**.

Dessa forma, conclui-se que o tráfego é um fator relevante no aumento do tempo de entrega.

####**Entrega X Horário**

In [ ]:
#Tabela de distribuição (categoria_entrega e horario)
tab_horario = pd.crosstab(df['horario'], df['categoria_entrega'])
tab_horario

In [ ]:
#Proporção
tab_horario_prop = pd.crosstab(df['horario'], df['categoria_entrega'], normalize = 'index') * 100
tab_horario_prop.round(2)

In [ ]:
#Distribuição
#Gráfico categoria_entrega X horario
sns.countplot(data=df, x='horario', hue='categoria_entrega', palette=['#0AF567', '#FDA615', '#F50201'])

plt.title('Horário x Entrega')
plt.xlabel('Horário')
plt.ylabel('Quantidade')
plt.legend(title='Tempo de Entrega')
plt.show()

In [ ]:
#Proporção
plt.figure(figsize=(16, 8))

colors = ['#0AF567', '#FDA615', '#F50201']
tab_horario_prop.plot(kind='bar', stacked=True, color=colors)

plt.title('Proporção do Tempo de Entrega pelo Horário')
plt.xlabel('Horário')
plt.ylabel('Percentual (%)')
plt.legend(title='Tempo de Entrega',
           bbox_to_anchor=(1.05, 1),
           loc='upper left')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Média de tempo de entrega por Horário
media_horario = df.groupby('horario')['tempo_entrega'].mean()
media_horario.round(2)

In [ ]:
#Boxplot
sns.boxplot(x='horario', y='tempo_entrega', data=df, color="#f4fa9c")

plt.title('Distribuição Tempo de Entrega por Horário')
plt.xlabel('Horário')
plt.ylabel('Tempo de Entrega (min)')
plt.xticks(rotation=0)
plt.show()

Após as análises do tempo de entrega nos três períodos do dia (**manhã, tarde** e **noite**), observa-se que a distribuição se mantém semelhante entre os diferentes horários. As entregas de tempo **médio** e **longo** são predominantes em todos os períodos.

Nota-se também que o tempo médio de entrega não apresenta grandes variações entre **manhã**, **tarde** e **noite**, com uma diferença de pouco mais de 1 minuto.

Com isso, conclui-se que o horário não possui uma influência significativa sobre o tempo da entrega.

####**Entrega X Tipo de veículo**

In [ ]:
#Tabela de distribuição (categoria_entrega e tipo_veiculo)
tab_veiculo = pd.crosstab(df['tipo_veiculo'], df['categoria_entrega'])
tab_veiculo

In [ ]:
#Proporção
tab_veiculo_prop = pd.crosstab(df['tipo_veiculo'], df['categoria_entrega'], normalize = 'index') * 100
tab_veiculo_prop.round(2)

In [ ]:
#Distribuição
#Gráfico categoria_entrega X tipo_veiculo
sns.countplot(data=df, x='tipo_veiculo', hue='categoria_entrega', palette=['#0AF567', '#FDA615', '#F50201'])

plt.title('Tipo de veículo x Entrega')
plt.xlabel('Tipo de veículo')
plt.ylabel('Quantidade')
plt.legend(title='Tempo de Entrega')
plt.show()

In [ ]:
#Proporção
plt.figure(figsize=(16, 8))

colors = ['#0AF567', '#FDA615', '#F50201']
tab_veiculo_prop.plot(kind='bar', stacked=True, color=colors)

plt.title('Proporção do Tempo de Entrega por Tipo de Veículo')
plt.xlabel('Veículo')
plt.ylabel('Percentual (%)')
plt.legend(title='Tempo de Entrega',
           bbox_to_anchor=(1.05, 1),
           loc='upper left')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Média de tempo de entrega por Tipo de Veículo
media_veiculo = df.groupby('tipo_veiculo')['tempo_entrega'].mean()
media_veiculo.round(2)

In [ ]:
#Boxplot
sns.boxplot(x='tipo_veiculo', y='tempo_entrega', data=df, color="#f4fa9c")

plt.title('Distribuição Tempo de Entrega por Tipo de veículo')
plt.xlabel('Veículo')
plt.ylabel('Tempo de Entrega (min)')
plt.xticks(rotation=0)
plt.show()

Com as análises do tempo de entrega com relação ao tipo de veículo utilizado (**carro**, **bicicleta** ou **motocicleta**), observa-se uma distribuição semelhante entre os diferentes tipos de veículo.

As entregas com tempo **médio** e **longo** são predominantes nos três tipos de veículo e o tempo médio de entrega não apresenta variações expressivas, pois a diferença é de aproximadamente 2 minutos.

Por isso, entende-se que o tipo de veículo utilizado, não possui influência significativa sobre o tempo da entrega.

####**Entrega X Preparo**

In [ ]:
#Tabela de distribuição (categoria_entrega e preparo)
tab_preparo = pd.crosstab(df['categoria_preparo'], df['categoria_entrega'])
tab_preparo

In [ ]:
#Proporção
tab_preparo_prop = pd.crosstab(df['categoria_preparo'], df['categoria_entrega'], normalize = 'index') * 100
tab_preparo_prop.round(2)

In [ ]:
#Distribuição
#Gráfico categoria_entrega X categoria_preparo
sns.countplot(data=df, x='categoria_preparo', hue='categoria_entrega', palette=['#0AF567', '#FDA615', '#F50201'])

plt.title('Preparo x Entrega')
plt.xlabel('Preparo')
plt.ylabel('Quantidade')
plt.legend(title='Tempo de Entrega')
plt.show()

In [ ]:
#Proporção
plt.figure(figsize=(16, 8))

colors = ['#0AF567', '#FDA615', '#F50201']
tab_preparo_prop.plot(kind='bar', stacked=True, color=colors)

plt.title('Proporção do Tempo de Entrega por Preparo')
plt.xlabel('Preparo')
plt.ylabel('Percentual (%)')
plt.legend(title='Tempo de Entrega',
           bbox_to_anchor=(1.05, 1),
           loc='upper left')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Média de tempo de entrega por Tipo de Veículo
media_preparo = df.groupby('categoria_preparo')['tempo_entrega'].mean()
media_preparo.round(2)

In [ ]:
#Boxplot
sns.boxplot(x='categoria_preparo', y='tempo_entrega', data=df, color="#f4fa9c")

plt.title('Distribuição Tempo de Entrega por Preparo')
plt.xlabel('Preparo')
plt.ylabel('Tempo de Entrega (min)')
plt.xticks(rotation=0)
plt.show()

Analisando o preparo e sua relação com o tempo de entrega, observa-se que a categoria de preparo **lento** não apresentou valores suficientes para análise. As entregas de tempo **médio** e **longo** são predominantes no preparo **rápido** e no **médio**.

O tempo médio do preparo apresentou uma variação de um pouco mais de 10 minutos, sendo de aproximadamente 50 minutos para o preparo **rápido** e 61 minutos para o preparo **médio**.

Esta análise indica que o tempo de preparo pode influenciar o tempo de entrega, sugerindo assim uma relação entre essas duas variáveis.




####**Entrega X Nível de Experiência**

In [ ]:
#Tabela de distribuição (categoria_entrega e nivel_experiencia)
tab_exp = pd.crosstab(df['nivel_experiencia'], df['categoria_entrega'])
tab_exp

In [ ]:
#Proporção
tab_exp_prop = pd.crosstab(df['nivel_experiencia'], df['categoria_entrega'], normalize = 'index') * 100
tab_exp_prop.round(2)

In [ ]:
#Distribuição
#Gráfico categoria_entrega X nivel_experiencia
sns.countplot(data=df, x='nivel_experiencia', hue='categoria_entrega', palette=['#0AF567', '#FDA615', '#F50201'])

plt.title('Experiência x Entrega')
plt.xlabel('Experiência')
plt.ylabel('Quantidade')
plt.legend(title='Tempo de Entrega')
plt.show()

In [ ]:
#Proporção
plt.figure(figsize=(16, 8))

colors = ['#0AF567', '#FDA615', '#F50201']
tab_exp_prop.plot(kind='bar', stacked=True, color=colors)

plt.title('Proporção do Tempo de Entrega pelo Nível de Experiência')
plt.xlabel('Experiência')
plt.ylabel('Percentual (%)')
plt.legend(title='Tempo de Entrega',
           bbox_to_anchor=(1.05, 1),
           loc='upper left')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#Média de tempo de entrega por Nível de Experiência
media_exp = df.groupby('nivel_experiencia')['tempo_entrega'].mean()
media_exp.round(2)

In [ ]:
#Boxplot
sns.boxplot(x='nivel_experiencia', y='tempo_entrega', data=df, color="#f4fa9c")

plt.title('Distribuição Tempo de Entrega pelo Nível de Experiencia')
plt.xlabel('Experiência')
plt.ylabel('Tempo de Entrega (min)')
plt.xticks(rotation=0)
plt.show()

As entregas com tempo **médio** e **longo** são predominantes nas três classificações de nível de experiência (**iniciante**, **intermediário** e **experiente**).

O tempo médio de entrega varia entre aproximadamente 55 minutos para entregadores experientes, e 60 minutos para iniciantes, indicando uma leve redução no tempo conforme o nível de experiência aumenta.

Portanto, o nível de experiência não se apresenta como uma variável determinante no tempo final de entrega, pois a diferença na média de tempo é relativamente pequena.


#**Conclusão da Análise Exploratória**

Ao finalizar a análise exploratória do conjunto, observa-se que três variáveis possuem influência sobre o tempo de entrega: **distância**, **tráfego** e **tempo de preparo**.

Entregas com distâncias mais longas estão associadas a maiores tempos de entregas, assim como o tráfego alto resulta em entregas mais demoradas. O tempo de preparo também apresentou influência no aumento do tempo final de entrega.

As variáveis **clima**, **horário**, **tipo de veículo** e **nível de experiência** não apresentaram indicativos de impacto relevante no tempo final de entrega.

Com isso, conclui-se que, para otimização das entregas, deve-se priorizar fatores relacionados à distância, ao tráfego no momento da entrega e ao tempo de preparo do produto.

#**Análise de Interação entre Fatores**

Nesta etapa, é analisada a interação entre as variáveis que apresentaram impacto significativo no tempo total de entrega. Assim, é possível entender o que acontece quando esses fatores ocorrem juntos, ou seja, o que ocorre quando a distância é longa e o tráfego é alto, por exemplo.

In [ ]:
tab_dist_trafego = pd.crosstab(
    [df['categoria_distancia'],
    df['trafego']], df['categoria_entrega'],
    normalize = 'index') * 100

tab_dist_trafego.round(2)

####**Interpretação da Análise de Interação**

Analisando a interação entre distância e tráfego, observa-se que o impacto no tempo final de entrega se torna mais intenso quando a **distância** é **longa** e o **tráfego alto**, apresentando um cenário crítico em que 78,79% das entregas são com tempo **longo**.

Por outro lado, o cenário mais favorável para entregas otimizadas é observado em **distâncias curtas** com **tráfego baixo**, que concentram maior proporção de entregas com tempo **curto**, pouco mais de 57%.

Portanto, nota-se que o aumento da **distância** e do **tráfego** contribui para o aumento do tempo de entrega, e, quando combinados, o efeito é ainda mais evidente.

#####**Salvando dataset tratado**

In [ ]:
df.to_csv('dataset_tratado.csv', index=False)